In [1]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

In [2]:
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,recall_score,f1_score,classification_report
from src.data_load import load_data
from src.preprocess import preprocess

In [3]:
seed=42
qubits=4

X_train, X_test, y_train, y_test= load_data()
X_train_scale,X_test_scale, X_train_pca, X_test_pca, pca=preprocess(X_train,X_test,qubits)

In [4]:
mm = MinMaxScaler(feature_range=(0, np.pi))
X_train_quantum = mm.fit_transform(X_train_pca)
X_test_quantum = np.clip(mm.transform(X_test_pca), 0, np.pi)

In [5]:
feature_map=ZZFeatureMap(feature_dimension=qubits,reps=2,entanglement="linear")

C:\Users\nonym\AppData\Local\Temp\ipykernel_15424\4196849272.py:1: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map=ZZFeatureMap(feature_dimension=qubits,reps=2,entanglement="linear")


In [6]:
kernel=FidelityQuantumKernel(feature_map=feature_map)
K_train = kernel.evaluate(X_train_quantum, X_train_quantum)
off_diag = K_train[~np.eye(len(K_train), dtype=bool)]
print("Kernel off-diagonal mean:", off_diag.mean(), "std:", off_diag.std())
K_test = kernel.evaluate(X_test_quantum, X_train_quantum)
print(K_train.shape,K_test.shape)

Kernel off-diagonal mean: 0.08365918540928713 std: 0.10928016128740085
(455, 455) (114, 455)


In [8]:
qsvm=SVC(kernel="precomputed",C=1.0)
qsvm.fit(K_train, y_train)
y_pred_q=qsvm.predict(K_test)

In [9]:
print("QSVM accuracy:", accuracy_score(y_test, y_pred_q))
print("Malignant recall:", recall_score(y_test, y_pred_q, pos_label=0))
print("F1 (macro):", f1_score(y_test, y_pred_q, average="macro"))
print(classification_report(y_test, y_pred_q, target_names=["malignant", "benign"]))

QSVM accuracy: 0.8596491228070176
Malignant recall: 0.7619047619047619
F1 (macro): 0.845945945945946
              precision    recall  f1-score   support

   malignant       0.84      0.76      0.80        42
      benign       0.87      0.92      0.89        72

    accuracy                           0.86       114
   macro avg       0.86      0.84      0.85       114
weighted avg       0.86      0.86      0.86       114



In [10]:
svm = SVC(kernel="rbf", C=1.0, gamma="scale").fit(X_train_quantum, y_train)
y_pred_classical = svm.predict(X_test_quantum)
print("Classical SVM (same data) accuracy:", accuracy_score(y_test, y_pred_classical))
print("Classical malignant recall:", recall_score(y_test, y_pred_classical, pos_label=0))
print("Classical F1 (macro):", f1_score(y_test, y_pred_classical, average="macro"))

Classical SVM (same data) accuracy: 0.956140350877193
Classical malignant recall: 0.9285714285714286
Classical F1 (macro): 0.9526381387619444
